# MapReduce con Spark: Auditoría de Ecosistema de Datos


Aplicamos el paradigma de procesamiento distribuido (MapReduce) mediante RDDs de Apache Spark. El objetivo es auditar un recorte de nuestro catálogo de Gobierno de Datos (extraído del DML transaccional).

In [ ]:
!pip install pyspark
from pyspark import SparkContext, SparkConf

# Inicializar el entorno Spark
conf = SparkConf().setAppName('TP_MapReduce_Gobierno').setMaster('local[*]')
sc = SparkContext.getOrCreate(conf=conf)
print('Spark Context iniciado correctamente.')

In [ ]:
# --- CARGA DE DATOS (Mapeo directo desde dml_gobierno_datos.sql) ---

# SUJETO: id_sujeto, tipo_sujeto
sujetos_rdd = sc.parallelize([
    '1000,Miembro UO', '1001,Miembro UO', '1002,Miembro UO',
    '2000,Externo', '2001,Externo', '3000,Organizacion', '3001,Organizacion'
])

# ACTIVO: id_activo, nombre, tipo_activo, id_fase, id_definicion
activos_rdd = sc.parallelize([
    '1,Reporte Ingreso Neto Mensual,Objeto,3,3',
    '26,Panel Ejecutivo Finanzas,Panel De Control,3,3',
    '46,ETL Ingesta Diaria Ventas,Objeto,2,3',
    '61,Tabla Maestra de Clientes,Datos Estructurado,2,1',
    '76,Archivos PDF de Contratos,Datos No Estructurado,2,5',
    '83,Modelo Predictivo Churn,Modelo,3,6'
])

# HERRAMIENTA: id_herramienta, nombre, tipo_herramienta, id_proveedor
herramientas_rdd = sc.parallelize([
    '1,SQL Server,Software,1', '4,Amazon S3,Licencia,2',
    '8,Apache Kafka,Herramienta Open Source,3', '13,Tableau Desktop,Software,4',
    '16,Snowflake,Software,5', '20,BigQuery,Software,7'
])

# PROBLEMA: id_problema, descripcion, fecha_origen, estado, id_fuente, id_activo
problemas_rdd = sc.parallelize([
    '1,Datos duplicados en tabla de clientes,2022-03-15,Resuelto,2,3',
    '8,Stock negativo en reporte de seguridad,2023-07-25,No Resuelto,3,9',
    '10,Timeout en query de tablero ejecutivo,2023-09-01,No Resuelto,1,36',
    '17,Proceso ETL de RRHH fallando los lunes,2024-01-22,No Resuelto,4,59'
])

# ACCESO_FUENTE_DATOS: id_sujeto, id_fuente, id_rol, id_perfil
accesos_rdd = sc.parallelize([
    '1000,1,1,4', '1000,3,2,4', '1001,1,2,4', '1001,2,2,4',
    '1008,8,3,1', '2000,1,3,2', '3000,1,5,5'
])

print('RDDs cargados en memoria listos para MapReduce.')

### Caso 1: Distribución del Capital Humano y Entidades (SUJETO)

Permite al equipo de Gobierno de Datos auditar la proporción de usuarios internos frente a externos u organizaciones que interactúan con el ecosistema. Es un indicador esencial para la asignación de licencias y para dimensionar la superficie de riesgo en las auditorías de seguridad perimetral.

* Fase Map: Se recorre cada registro del RDD, extrayendo el tipo_sujeto como clave y asignándole el valor 1 (patrón clave-valor).
*   Fase Reduce: Se agrupan las claves idénticas y se suman sus valores (a + b), obteniendo la frecuencia total por tipo.

* Fase Recolección: Se ejecuta la acción collect() para traer los resultados distribuidos al nodo maestro y mostrarlos en consola.




In [ ]:
# La Fase de Map
map_sujetos = sujetos_rdd.map(lambda x: (x.split(',')[1], 1))

# La Fase de Reduce
reduce_sujetos = map_sujetos.reduceByKey(lambda a, b: a + b)

# La Fase de Recolección de datos e Impresión
print('1. Tipos de Sujetos operando en el Gobierno de Datos:')
print(reduce_sujetos.collect())

**Análisis:** La plataforma cuenta con 3 usuarios internos. Sin embargo, la suma de entidades externas y organizaciones (4) supera a los miembros internos, justificando la necesidad de aplicar, en un entorno real, políticas de auditoría perimetral estrictas.

### Caso 2: Madurez del Catálogo por Fase de Vida (Tabla ACTIVO)

Mide la salud operativa de los despliegues de datos. Permite identificar si existen cuellos de botella organizacionales; por ejemplo, si muchos activos están estancados en fase de "Creación" en lugar de estar aportando valor en fases operativas.

*  Fase Map: El clúster procesa cada registro del RDD de la tabla ACTIVO separando sus campos. Se extrae específicamente el id_fase (índice 3) para utilizarlo como clave de agrupación, emitiendo el par clave-valor (id_fase, 1).
*   Fase Reduce: Mediante la transformación reduceByKey, los nodos de procesamiento agrupan de forma distribuida todos los registros que comparten la misma clave (la misma fase de vida). Luego, aplican una suma acumulativa (a + b) calculando la volumetría total por cada etapa.
* Fase Recolección: Se ejecuta la acción collect() para traer los resultados distribuidos al nodo maestro y mostrarlos en consola.

In [ ]:
# La Fase de Map
map_activos = activos_rdd.map(lambda x: (x.split(',')[3], 1))

# La Fase de Reduce
reduce_activos = map_activos.reduceByKey(lambda a, b: a + b)

# La Fase de Recolección de datos e Impresión
print('\n2. Cantidad de Activos por Fase de Vida (ID Fase):')
print(reduce_activos.collect())

**Análisis:** El inventario simulado se encuentra balanceado equitativamente entre las fases 2 (Almacenamiento) y 3 (Uso). Esto indica un ciclo de desarrollo saludable donde los activos logran superar las etapas tempranas y llegan a ser consumidos.

### Caso 3: Arquitectura Tecnológica (Tabla HERRAMIENTA)

Permite realizar un inventario automatizado del tipo de tecnologías que soportan el catálogo (Software comercial, Licencias, Open Source). Esto es clave para el control de costos de IT, auditorías de obsolescencia tecnológica y la detección de dependencias críticas de proveedores.

*   Fase Map: Se procesa el RDD de herramientas dividiendo las cadenas por comas. Se extrae el tipo_herramienta (índice 2) como clave y se le asocia el valor constante 1 para iniciar el conteo.
*   Fase Reduce: Mediante la transformación reduceByKey(lambda a, b: a + b), Spark agrupa los nodos que procesan el mismo tipo de herramienta y consolida la suma total de elementos por categoría.
*   Fase Recolección: Se aplica la acción collect() para consolidar los resultados particionados en el Driver e imprimir en pantalla la distribución actual del stack de datos.

In [ ]:
# La Fase de Map
map_herramientas = herramientas_rdd.map(lambda x: (x.split(',')[2], 1))

# La Fase de Reduce
reduce_herramientas = map_herramientas.reduceByKey(lambda a, b: a + b)

# La Fase de Recolección de datos e Impresión
print('\n3. Distribucion del Stack Tecnologico:')
print(reduce_herramientas.collect())

**Análisis:** Existe una clara dependencia funcional sobre el Software propietario y Licencias (5 en total), frente a una adopción mínima de herramientas Open Source (1). Esto representa, en entorno real, una oportunidad gerencial para reestructurar costos IT en futuras migraciones.

### Caso 4: Balance de Calidad de Datos (Tabla PROBLEMA)

Este caso es clave para la resolución de incidentes de la organización. Permite medir el desempeño de los Data Stewards y los equipos resolutores, contrastando el volumen de datos sanos ("Resueltos") frente a la deuda técnica latente ("No Resueltos").

*   Fase Map: Se itera sobre el RDD de incidentes operacionales, mapeando el campo estado (índice 3) como la clave semántica y emitiendo un 1 por cada ocurrencia detectada.
*   Fase Reduce: La función de reducción realiza una agregación asociativa y conmutativa sumando los enteros por cada estado único del ciclo de vida del problema.
*   Fase Recolección: La acción collect() recupera los pares clave-valor calculados en forma distribuida para su visualización y evaluación gerencial.

In [ ]:
# La Fase de Map
map_problemas = problemas_rdd.map(lambda x: (x.split(',')[3], 1))

# La Fase de Reduce
reduce_problemas = map_problemas.reduceByKey(lambda a, b: a + b)

# La Fase de Recolección de datos e Impresión
print('\n4. Estado global de los incidentes de calidad:')
print(reduce_problemas.collect())

**Análisis:** La deuda técnica (No Resuelto) triplica a los incidentes solucionados en la muestra actual. Este KPI, en entorno real, evidenciaría que el equipo de soporte se encuentra saturado o que existen bloqueantes técnicos severos que demoran la resolución.

### Caso 5: Auditoría de Permisos por Nivel de Perfil (Tabla ACCESO_FUENTE_DATOS)

Este caso es crucial para el pilar de seguridad de datos. Al agrupar las autorizaciones por id_perfil, el Oficial de Protección de Datos (DPO) puede auditar si se cumple el Principio de Menor Privilegio. Permite detectar anomalías de sobre-otorgamiento de perfiles de alta criticidad (ej. Escritura/Administración) frente a accesos restrictivos (Lectura).
*   Fase Map: Se tokeniza el RDD de accesos perimetrales y se extrae el id_perfil (índice 3) para actuar como clave, emitiendo un 1 asociado.
*   Fase Reduce: Se colacionan de manera distribuida las claves numéricas de los perfiles y se acumulan (a + b) para dimensionar cuántos accesos activos posee cada nivel de privilegios.
*   Fase Recolección: La llamada a collect() finaliza el pipeline del RDD trayendo la matriz estructurada al nodo central para su impresión.

In [ ]:
# La Fase de Map
map_accesos = accesos_rdd.map(lambda x: (x.split(',')[3], 1))

# La Fase de Reduce
reduce_accesos = map_accesos.reduceByKey(lambda a, b: a + b)

# La Fase de Recolección de datos e Impresión
print('\n5. Cantidad de accesos otorgados agrupados por ID Perfil:')
print(reduce_accesos.collect())

**Análisis:** Existe una sobre-asignación de accesos correspondientes al Perfil 4 (Alta jerarquía/Administrador según el catálogo). Esto en un entorno real, representaría un riesgo perimetral que el Oficial de Seguridad (CISO) debe auditar y revocar inmediatamente.

### Caso 6: Monitoreo de Vulnerabilidad - Fuentes Críticas (Tablas PROBLEMA)

Prioriza la atención del equipo de Data Engineering al identificar qué fuentes de datos físicas concentran la mayor cantidad de incidentes "No Resueltos".



*   Fase de Filtrado y Map: Primero se aplica una transformación filter para descartar los problemas resueltos. Luego, la fase Map extrae el id_fuente afectado y le asigna el valor 1.
*   Fase Reduce: Se suman las incidencias por cada fuente física.
*   Fase Recolección: collect() devuelve el ranking de sistemas críticos para su remediación inmediata.

In [ ]:
# Usamos un filter previo y luego aplicamos la Fase de Map
map_fuentes_criticas = problemas_rdd.filter(lambda x: x.split(',')[3] == 'No Resuelto').map(lambda x: (x.split(',')[4], 1))

# La Fase de Reduce
reduce_fuentes_criticas = map_fuentes_criticas.reduceByKey(lambda a, b: a + b)

# La Fase de Recolección de datos e Impresión
print('\n6. Fuentes con incidentes No Resueltos (ID Fuente : Cantidad):')
print(reduce_fuentes_criticas.collect())

**Análisis:** La reducción aisló la deuda técnica y determinó que las fuentes físicas 1, 3 y 4 requieren mantenimiento prioritario. Esto en un entorno real provee un diagnóstico directo y exacto para que el equipo de Infraestructura focalice sus esfuerzos de remediación.